# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and processing of the FAIR^2 dataset using the `mlcroissant` library, referencing each entity by its `@id`.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the FAIR^2 Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Identifier (@id): {metadata['@id']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all RecordSet entities and their @ids
record_sets = []
for obj in dataset.metadata.to_json().get('hasPart', []):
    if obj.get('@type') == 'RecordSet':
        record_sets.append(obj['@id'])

if not record_sets:
    # Fallback: Try to use field named 'recordSet' (per schema)
    record_sets = dataset.metadata.to_json().get('recordSet', [])

if record_sets:
    print('RecordSets available:')
    for rs_id in record_sets:
        print(f'  - {rs_id}')
else:
    print('No record sets explicitly listed; attempting generic access.')

# Show sample records and field @ids for each available record set
for rs_id in record_sets:
    print(f'\nSample records from recordSet @id: {rs_id}')
    try:
        sample_records = list(dataset.records(record_set=rs_id))
        if sample_records:
            sample = sample_records[0]
            print(f'Record fields (@id): {list(sample.keys())}')
            print('Sample record:', sample)
        else:
            print('  No records found.')
    except Exception as e:
        print(f'  Could not list records for {rs_id}: {e}')

# If no record sets found, attempt to enumerate any available root-level RecordSets
if not record_sets:
    try:
        test_rs = '__all__'
        sample_records = list(dataset.records(record_set=test_rs))
        print(f'\nSample records from recordSet @id: {test_rs}')
        if sample_records:
            sample = sample_records[0]
            print(f'Record fields (@id): {list(sample.keys())}')
            print('Sample record:', sample)
        else:
            print('  No records found.')
    except Exception as e:
        print(f'  Could not list records for {test_rs}: {e}')

## 3. Data Extraction
Load data from each record set into a pandas DataFrame using the record set and field `@id`s.

In [ ]:
dataframes = {}

if record_sets:
    for rs_id in record_sets:
        print(f'Loading records for recordSet @id: {rs_id}')
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f'Columns (@id): {df.columns.tolist()}')
        print(df.head(2))
else:
    # Fallback: Attempt loading from '__all__'
    records = list(dataset.records(record_set='__all__'))
    df = pd.DataFrame(records)
    dataframes['__all__'] = df
    print(f'Columns (@id): {df.columns.tolist()}')
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric fields, categorize/group, referencing fields by their `@id`.

For demonstration, we use the first available record set and numeric field.

In [ ]:
# Select the first record set and its numeric column for demonstration
selected_rs_id = next(iter(dataframes.keys()))
df = dataframes[selected_rs_id]

# Try to automatically detect a numeric field
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is not None:
    print(f"Using numeric field (@id): {numeric_field_id}")
    threshold = df[numeric_field_id].mean()

    # Filter records based on threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a categorical field
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by field (@id): {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
else:
    print("No numeric field found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset referencing their `@id`s.

In [ ]:
# Basic histogram of the numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot if categorical group
    if group_field_id:
        plt.figure(figsize=(6,4))
        plt.scatter(df[group_field_id], df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} vs {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated data loading, overview, processing, and visualization with `mlcroissant` referencing all entities by their `@id`. The FAIR^2 dataset comprises clinical, hormone, and genetic records for three NC-21OHD patients. Due to the limited sample size and scope, analysis is mainly descriptive; all processing referenced schema entity `@id`s for clarity and reproducibility.